# 4.1 Landmark-based models

In order to carry out primary tasks like localization or navigation, or more high-level tasks like interacting with humans, a mobile robot has to **perceive** its workspace. A variety of sensors can be used for that, as well as a number of probabilistic models for managing their behavior.

Typically, the sensors used onboard the robot do not deliver the exact truth of the quantities they are measuring, but a perturbed version. This is due to the working (physical) principles that govern the sensors behavior, and to the conditions of their workspaces (illumination, humidity, temperature, etc.). 

As an illustrative example of this, there is a popular European company called [SICK](https://www.sick.com/es/es/), which develops 2D LiDAR sensors (among other devices). One of its most popular sensors is the [TiM2xx one](https://www.sick.com/gb/en/catalog/products/lidar-and-radar-sensors/lidar-sensors/tim/tim240-2050300/p/p654443) (see left part of Fig. 1), which can be easily integrated into a robotic platform. If we take a look at the specifications about the performance of such device, we can check how this uncertainty about the sensor measurements is explicitly specified (systematic error and statistical error), as well as how these values depend on environmental conditions (see right part of Fig. 1). $\\[10pt]$

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/sick-laser.PNG" alt="">
  <figcaption>Fig. 1. Left, TiM2xx sensor from SICK. Right, performance details of such sensor.</figcaption>
</figure>    

$\\[5pt]$ To account for this behavior, sensors' measurements in probabilistic robotics will be modeled by... wait for it... a probability distribution: the measurement is a random variable $Z \sim p(z|v)$, where $z$ is one particular measurement (a realization of $Z$) and $v$ is the true (unknown) value. The sensor error is often modeled as Gaussian, that is, $p(z|v) = \mathcal{N}(z; v, \sigma^2)$, being $\sigma^2$ the measurement variance.

## 4.1.1 Dealing with landmark-based models

In different applications it is interesting for the robot to detect **landmarks** in its workspace and build internal representations of them, commonly referred to as maps. A landmark can be defined as a distinctive feature present in the environment, that can be used to perform localization, map building, or navigation, since they provide a fixed reference point in the environment. They can be of different nature:

- **Natural landmarks**: mountains, trees, rivers, rocks, etc.
- **Artificial landmarks**: buildings, signs, traffic lights, doors, windows, furniture, etc.
  - ***Purpose-built landmarks***: QR codes, [ArUco](https://www.uco.es/investiga/grupos/ava/portfolio/aruco/) codes, visual-patterns in general, RFID tags, beacons, etc.

Fig. 2 shows some examples of each type.
 
<center>
    <a id="figure"></a>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/types-of-landmarks.png" width="800">
    <figcaption>Fig. 2. Types of landmarks: natural, artificial and purpose-built (e.g. the pattern of three black circles used to locate a recharging station).</figcaption>
</center>
</br>
 
In both cases, landmarks or features like corners, blobs, etc. could also be extracted, e.g. using a camera.

In the case of maps consisting of a collection of landmarks, $\mathcal{M} = \mathbf{m} = \{\mathbf{m}_i\},\ i=1,\dots,N$, where $\mathbf{m}_i = [x_i, y_i]^T$ is the position of the $i$-th landmark, different types of sensors can be used to provide observations $\mathbf{z}_i$ of those landmarks:

- **Distance/range** (*e.g.* radio, GPS, etc.): $\\[2pt]$
  $\hspace{1cm}$ $z_i = r_i = h_i(\mathbf{x},\mathbf{m})+w_i$ $\\[2pt]$
- **Bearing** (*e.g.* camera): $\\[2pt]$
  $\hspace{1cm}$ $z_i = \alpha_i = h_i(\mathbf{x},\mathbf{m})+w_i$ $\\[2pt]$
- **Distance/range and bearing** (*e.g.* stereo, features in a scan, etc.) $\\[2pt]$
  $\hspace{1cm}$ $\mathbf{z}_i = [r_i,\alpha_i]^T = \mathbf{h}_i(\mathbf{x},\mathbf{m})+\mathbf{w}_i$ *(in this case, $\mathbf{h}_i(\mathbf{x},\mathbf{m})$ and $\mathbf{w}_i$ are 2D vectors)* $\\[2pt]$

where:

- $\mathbf{z}_i$ is an observation, $\mathbf{x}$ is the sensor pose, and $\mathbf{m}$ is the map of the environment (here, the set of landmarks),
- $\mathbf{h}(\mathbf{x},\mathbf{m})$ is the Observation (or measurement, or prediction) function: it predicts the value of the observation $\mathbf{z}_i$ given the state values $\mathbf{x}$ and $\mathbf{m}$, that is, the *ideal* measurement $\mathbf{v} = \mathbf{h}(\mathbf{x},\mathbf{m})$, and
- $\mathbf{w}$ is the measurement error, modeled by a zero-mean Gaussian distribution $\mathbf{w} \sim \mathcal{N}(\mathbf{0},Q)$, being $Q$ its covariance matrix.

In this way, the probability distribution $p(\mathbf{z}|\mathbf{x},\mathbf{m})$ modeling the sensor measurements results:

$$
p(\mathbf{z}|\mathbf{x},\mathbf{m}) = \mathcal{N}(\mathbf{z}; \mathbf{h}(\mathbf{x},\mathbf{m}), Q) = K \exp\{-\frac{1}{2}[\mathbf{h}(\mathbf{x},\mathbf{m})-\mathbf{z}]^T Q^{-1} [\mathbf{h}(\mathbf{x},\mathbf{m})-\mathbf{z}]\}
$$

where $K = 1/\sqrt{(2\pi)^k |Q|}$ is a normalization constant (it makes the pdf integrate to 1) and $k$ is the dimension of $\mathbf{z}$.

Recall that this probability is used, for example, to update our belief about the robot pose $\mathbf{x}$ after receiving an observation $\mathbf{z}$, by means of Bayes' theorem:

$$
p(\mathbf{x}|\mathbf{z},\mathbf{m}) = \frac{p(\mathbf{z}|\mathbf{x},\mathbf{m})\, p(\mathbf{x}|\mathbf{m})}{p(\mathbf{z}|\mathbf{m})} \propto p(\mathbf{z}|\mathbf{x},\mathbf{m})\, p(\mathbf{x}|\mathbf{m})
$$

In the localization chapter, this will be extended to the Bayes filter, which also takes the robot motion into account.

These types of maps and sensor measurements pose a new problem: **data association**, that is, which landmark $\mathbf{m}_i$ the observation $\mathbf{z}_i$ corresponds to:

$$\mathbf{h}_i(\mathbf{x},\mathbf{m})=\mathbf{h}(\mathbf{x},\mathbf{m}_i)$$

This problem is usually addressed by using the landmark descriptor (e.g. its visual features) or by applying Chi-squared tests, although for the sake of simplicity in this course we will consider it as solved. 

### Playing with landmarks and robot poses

In the rest of this section we will familiarize ourselves with the process of observing landmarks from robots located at certain poses, as well as the transformations needed to make use of these observations, that is, to express those observations into the world frame and backwards.

Some relevant concepts:

- **World frame**: $(x, y)$ coordinates from a selected point of reference $(0, 0)$ used to keep track of the robot's pose and landmarks within the map.
- **Observation**: Information from the real world provided by a sensor, from the point of view (*pov*) of a certain robot.
- **Range-bearing sensor**: Sensor model being used in this lesson. This kind of sensors detect how far an object is (its range $r$) and its orientation relative to the robot's one (its bearing $\alpha$), that is, they provide observations $\mathbf{z} = [r, \alpha]^T$.

The main tools to deal with those concepts are:

- the composition of two poses.
- the composition of a pose and a landmark.
- the propagation of uncertainty through the Jacobians of these compositions.

We will address several problems of incremental complexity. In all of them, it is important to have in mind how the composition of a (robot) pose and a landmark point works:


<center>
    <a id="figure"></a>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/fig4-1-pose-landmark-composition.png">
    <figcaption>Fig. 3. Composition of a pose and a landmark point.</figcaption>
</center>

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Change directory to this chapter's folder in your Drive copy of the course repo:
    %cd '/content/drive/MyDrive/Colab Notebooks/uma_robotics_2027/Chapter 04. Robot sensing'
except ImportError:
    pass

#%matplotlib widget
%matplotlib inline

# IMPORTS

import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("..")
from utils.PlotEllipse import PlotEllipse
from utils.DrawRobot import DrawRobot
from utils.tcomp import tcomp
from utils.tinv import tinv, jac_tinv1 as jac_tinv
from utils.Jacobians import J1, J2

### **<span style="color:green"><b><i>ASSIGNMENT 1: Expressing an observed landmark in coordinates of the world frame</i></b></span>** <a id="4211"></a>

Let’s consider a robot R1 at a perfectly known pose $\mathbf{x}_1 = [1, 2, 0.5]^T$ (no uncertainty at this point) which observes a landmark $\mathbf{m}$ with a range-bearing (polar) sensor. The sensor provides the measurement $[4\text{ m}, 0.7\text{ rad}]^T$ and is affected by a zero-mean Gaussian error with covariance $\Sigma_{\mathbf{z}^1_p} = \text{diag}([0.25, 0.04])$ (in the robot frame). Therefore, we model the observation as a Gaussian centered at the measured value: $\mathbf{z}^1_p \sim \mathcal{N}(\bar{\mathbf{z}}^1_p, \Sigma_{\mathbf{z}^1_p})$, with mean $\bar{\mathbf{z}}^1_p = [4\text{ m}, 0.7\text{ rad}]^T$. The scenario is the one in Fig. 4.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/assignment_1_scenario.png"/>
    <figcaption>Fig. 4. Illustration of the scenario in assignment 1.</figcaption>
</center>

**You are tasked to** compute the Gaussian probability distribution $\mathbf{m}^1 \sim \mathcal{N}(\bar{\mathbf{m}}^1, \Sigma_{\mathbf{m}^1})$ (mean and covariance) of the landmark observation in the world frame (the same as the robot) and plot its corresponding ellipse (in green, $\sigma=1$). Concretely, you have to complete the ``to_world_frame()`` function, and modify the demo code to show the ellipse representing the uncertainty.

To do so, recall that the landmark position in the world frame is the composition of the robot pose, $\mathbf{x}_1$, and the landmark observation expressed in Cartesian coordinates in the robot frame, $\mathbf{z}^1_c$:

$$
\mathbf{m}^1 = \mathbf{x}_1 \oplus \mathbf{z}^1_c = \mathbf{f}(\mathbf{x}_1, \mathbf{z}^1_c)
$$

Then, by first-order uncertainty propagation (that is, linearizing $\mathbf{f}$ around the means), the mean and the covariance of the landmark in the world frame are:

$$
\bar{\mathbf{m}}^1 = \bar{\mathbf{x}}_1 \oplus \bar{\mathbf{z}}^1_c
$$

$$
\Sigma_{\mathbf{m}^1} = \frac{\partial \mathbf{m}^1}{\partial \mathbf{x}_1} \ \Sigma_{\mathbf{x}_1} \ \left( \frac{\partial \mathbf{m}^1}{\partial \mathbf{x}_1} \right)^T +
     \frac{\partial \mathbf{m}^1}{\partial \mathbf{z}^1_c} \ \Sigma_{\mathbf{z}^1_c} \ \left( \frac{\partial \mathbf{m}^1}{\partial \mathbf{z}^1_c} \right)^T
$$

where $\Sigma_{\mathbf{x}_1}$ is the covariance of the robot pose $\mathbf{x}_1$ (zero in this assignment), and $\Sigma_{\mathbf{z}^1_c}$ is the covariance of the observation in Cartesian coordinates. Note that this expression assumes that the robot pose $\mathbf{x}_1$ and the observation $\mathbf{z}^1_c$ are independent (uncorrelated), which is why their contributions to $\Sigma_{\mathbf{m}^1}$ simply add up.

Mathematical pill: the Jacobian of a function $\mathbf{f}(\mathbf{a}) = [f_1, \dots, f_k]^T$ of $n$ variables $\mathbf{a} = [a_1, \dots, a_n]^T$ is the $k \times n$ matrix:

$$
\frac{\partial \mathbf{f}}{\partial \mathbf{a}} =
\begin{bmatrix}
    \frac{\partial f_1}{\partial a_1} & \cdots & \frac{\partial f_1}{\partial a_n} \\
    \vdots & \ddots & \vdots \\
    \frac{\partial f_k}{\partial a_1} & \cdots & \frac{\partial f_k}{\partial a_n}
\end{bmatrix}
$$

The Jacobians $\frac{\partial \mathbf{m}^1}{\partial \mathbf{x}_1}$ ($2 \times 3$) and $\frac{\partial \mathbf{m}^1}{\partial \mathbf{z}^1_c}$ ($2 \times 2$) are the same ones previously used to compose two poses in *robot motion*, but with a reduced size since **while working with landmarks the orientation is meaningless, only the position matters**. The functions ``J1()`` and ``J2()`` implement these Jacobians for you.

However, the sensor does not provide $\mathbf{z}^1_c$ and $\Sigma_{\mathbf{z}^1_c}$, but the observation in polar coordinates, $\mathbf{z}^1_p$, and its covariance $\Sigma_{\mathbf{z}^1_p}$. So, before composing, you have to express the observation in Cartesian coordinates. A sensor measurement in polar coordinates ($\mathbf{z}_p=[r,\alpha]^T$) is expressed in Cartesian coordinates ($\mathbf{z}_c=[z_x,z_y]^T$) by:

$$
\mathbf{z}_c = \begin{bmatrix} z_x \\ z_y \end{bmatrix}
    = \begin{bmatrix} r \cos\alpha \\ r \sin\alpha \end{bmatrix}
    = \mathbf{f}(\mathbf{z}_p)
$$

and its covariance is obtained, again, by propagating the uncertainty, in this case through the Jacobian of this function:

$$
\Sigma_{\mathbf{z}^1_c} = \frac{\partial \mathbf{z}_c}{\partial \mathbf{z}_p} \ \Sigma_{\mathbf{z}^1_p} \ \left( \frac{\partial \mathbf{z}_c}{\partial \mathbf{z}_p} \right)^T
$$

Example: $\\[5pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/result_1.PNG" />
    <figcaption>Fig. 5. Pose of a robot (without uncertainty) and position of an observed landmark with its associated uncertainty.</figcaption>
</center>

In [ ]:
def to_world_frame(x1_w, Qx1_w, z1_p_r, W1):
    """ Convert the observation z1_p_r to the world frame
    
        Args:
            x1_w: Pose of the robot x_1 (in world frame)
            Qx1_w: Covariance Sigma_x_1 of the robot pose
            z1_p_r: Observation z^1_p of a landmark (polar coordinates) from the robot's perspective
            W1: Covariance Sigma_z^1_p of the sensor in polar coordinates
    
        Returns:
            z1_w: Position of the landmark m^1 in the world frame (its mean)
            Wz1: Covariance Sigma_m^1 associated to z1_w
    """
    
    # Definition of useful variables
    r, a = z1_p_r[0,0], z1_p_r[1,0]
    s, c = np.sin(a), np.cos(a)

    # Jacobian dz_c/dz_p to convert the measurement uncertainty from polar to cartesian coordinates
    Jac_pol_car = np.array([
        [None, None],
        [None, None]
    ])

    # Build a tuple with:
    # z1_car_rel[0]: coordinates of the sensor measurement in cartesian coordinates relative to robot position (z^1_c)
    # z1_car_rel[1]: its associated uncertainty expressed in cartesian coordinates (Sigma_z^1_c)
    z1_car_rel = (
            np.vstack([None,None]), # position
            None@W1@None # uncertainty
            )
    
    z1_ext = np.vstack([z1_car_rel[0], 0]) # Extends z1 for its usage in the Jacobian functions J1 and J2

    # Build the Jacobians 
    Jac_ap = J1(x1_w ,z1_ext)[0:2,:] # Jacobian dm^1/dx_1 for expressing the uncertainty in the robot pose in a global frame
    Jac_aa = J2(x1_w ,z1_ext)[0:2,0:2] # This one (dm^1/dz^1_c) expresses the uncertainty in the measurement in a global frame
    
    z1_w = tcomp(None ,None)[0:2,[0]] # Compute coordinates of the landmark in the world
    Wz1 = (None @ Qx1_w @ None
          + None @ z1_car_rel[1] @ None) # Finally, propagate the covariance!
    
    return z1_w, Wz1

In [ ]:
# Robot
x1_w = np.vstack([1, 2, 0.5]) # Robot R1 pose x_1
Qx1_w = np.zeros((3, 3)) # Robot pose covariance matrix (uncertainty) Sigma_x_1

# Landmark observation
z1_p_r = np.vstack([4., .7]) # Measurement/Observation z^1_p
W1 = np.diag([0.25, 0.04]) # Sensor noise covariance Sigma_z^1_p

# Express the landmark observation in the world frame (mean and covariance of m^1)
z1_w, Wz1 = to_world_frame(x1_w, Qx1_w, z1_p_r, W1)

# Visualize the results
fig, ax = plt.subplots()
plt.xlim([-.5, 10])
plt.ylim([-.5, 10])
plt.grid()
plt.tight_layout()

DrawRobot(fig, ax, x1_w, label='R1', color='blue')
    
ax.plot(z1_w[0, 0], z1_w[1, 0], 'o', label='Z1', color='green')
PlotEllipse(fig, ax, None, None, color='green')

plt.legend()
print('----\tExercise 4.1.1\t----\n'+
      'z1_w = {}\'\n'.format(z1_w.flatten())
      + 'Wz1_w = \n{}\n'.format(Wz1))

<span style="color:blue">Expected results for demo:</span>


```
---- Exercise 4.1.1 ----
z1_w = [2.44943102 5.72815634]'
Wz1_w = 
[[ 0.58879177 -0.13171532]
 [-0.13171532  0.30120823]]
```

### **<span style="color:green"><b><i>ASSIGNMENT 2: Adding uncertainty to the robot position</i></b></span>**

Now, let’s assume that the robot pose is not known, but it is a RV that follows a Gaussian probability distribution: $\mathbf{x}_1 \sim \mathcal{N}(\bar{\mathbf{x}}_1, \Sigma_{\mathbf{x}_1})$, with $\bar{\mathbf{x}}_1 = [1, 2, 0.5]^T$ and $\Sigma_{\mathbf{x}_1} = \text{diag}([0.08, 0.6, 0.02])$.

1. Compute the covariance matrix $\Sigma_{\mathbf{m}^1}$ of the landmark in the world frame and plot it as an ellipse centered at the mean $\bar{\mathbf{m}}^1$ (in green, $\sigma = 1$). Plot also the covariance of the robot pose (in blue, $\sigma = 1$).

2. Compare the covariance with that obtained in the previous case.  $\\[5pt]$

Example: $\\[5pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/result_2.PNG" />
    <figcaption>Fig. 6. Pose of a robot and position of an observed landmark, along with their associated uncertainties.</figcaption>
</center>

In [ ]:
# Robot
x1_w = np.vstack([1, 2, 0.5]) # Robot R1 pose x_1
Qx1_w = np.diag([0.08,0.6,0.02])  # Robot pose covariance matrix (uncertainty) Sigma_x_1

# Landmark observation
z1_p_r = np.vstack([4., .7]) # Measurement/Observation z^1_p
W1 = np.diag([0.25, 0.04]) # Sensor noise covariance Sigma_z^1_p

# Express the landmark observation in the world frame (mean and covariance of m^1)
z1_w, Wz1 = to_world_frame(x1_w, Qx1_w, z1_p_r, W1)

# MATPLOTLIB
fig, ax = plt.subplots()
plt.xlim([-.5, 10])
plt.ylim([-.5, 10])
plt.grid()
plt.tight_layout()

fig.canvas.draw()

DrawRobot(fig, ax, x1_w, label='R1', color='red')  
PlotEllipse(fig, ax, None, None, color='blue')

ax.plot(z1_w[0, 0], z1_w[1, 0], 'o', label='Z1', color='green')
PlotEllipse(fig, ax, None, None, color='green')

plt.legend()
print('---- Exercise 4.1.2 ----\n'+
      'Wz1_w = \n{}\n'.format(Wz1))


<span style="color:blue">Expected results for demo:</span>

```
---- Exercise 4.1.2 ----
Wz1_w = 
[[ 0.94677477 -0.23978943]
 [-0.23978943  0.94322523]]
```

### **<span style="color:green"><b><i>ASSIGNMENT 3: Combining observations of the same landmark</i></b></span>**

Assume now that the robot moves to a new pose $\mathbf{x}_2 \sim \mathcal{N}(\bar{\mathbf{x}}_2, \Sigma_{\mathbf{x}_2})$, with $\bar{\mathbf{x}}_2 = [6\text{ m}, 4\text{ m}, 2.1\text{ rad}]^T$ and $\Sigma_{\mathbf{x}_2} = \text{diag}([0.20, 0.09, 0.03])$, and takes a measurement $[4\text{ m}, 0.3\text{ rad}]^T$ of the landmark with the same sensor, so the observation is $\mathbf{z}^2_p \sim \mathcal{N}(\bar{\mathbf{z}}^2_p, \Sigma_{\mathbf{z}^2_p})$, with $\bar{\mathbf{z}}^2_p = [4\text{ m}, 0.3\text{ rad}]^T$ and $\Sigma_{\mathbf{z}^2_p} = \Sigma_{\mathbf{z}^1_p}$. **You have to**:

1. Use the previously implemented `to_world_frame()` function to compute the position and uncertainty of the landmark from both measurements in the world frame, that is, $\mathbf{m}^1 \sim \mathcal{N}(\bar{\mathbf{m}}^1, \Sigma_{\mathbf{m}^1})$ and $\mathbf{m}^2 \sim \mathcal{N}(\bar{\mathbf{m}}^2, \Sigma_{\mathbf{m}^2})$.
2. Plot the robots and the two measurements along with their uncertainty (ellipses) in the world frame.
3. Combine both observations within the `combine_pdfs()` function, and show the resultant combined observation $\mathbf{m} \sim \mathcal{N}(\bar{\mathbf{m}}, \Sigma_{\mathbf{m}})$ along with its associated uncertainty.

Hint: since both observations are independent, the pdf that combines them is proportional to the product of their pdfs:

$$
p(\mathbf{m}) \propto \mathcal{N}(\mathbf{m};\, \bar{\mathbf{m}}^1, \Sigma_{\mathbf{m}^1})\; \mathcal{N}(\mathbf{m};\, \bar{\mathbf{m}}^2, \Sigma_{\mathbf{m}^2})
$$

and the product of two Gaussian pdfs is, up to a normalization constant, another Gaussian pdf $\mathcal{N}(\mathbf{m};\, \bar{\mathbf{m}}, \Sigma_{\mathbf{m}})$. Find out how its mean $\bar{\mathbf{m}}$ and covariance $\Sigma_{\mathbf{m}}$ are computed from those of the two observations, and implement it in `combine_pdfs()`.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/fig4-1-2.PNG" width="400" alt="">
  <figcaption>Fig. 7. Poses of R1 and R2, the two observations of the landmark (Z1, Z2) and their combination (Z3), with their uncertainty ellipses.</figcaption>
</figure>    

In [ ]:
def combine_pdfs(z1_w, Wz1_w, z2_w, Wz2_w):
    """ Method to combine the pdfs associated with two observations of the same landmark.  
        
        Args:            
            z1_w: Landmark m^1 observed in cartesian coordinates (world frame) from Robot 1
            Wz1_w: Covariance Sigma_m^1 associated to the landmark.
            z2_w: Landmark m^2 observed in cartesian coordinates (world frame) from Robot 2
            Wz2_w: Covariance Sigma_m^2 associated to the landmark.
        Returns:
            z: Combined observation m (its mean)
            W_z: Covariance Sigma_m associated to z
    """
    None # Implement the needed code here

    W_z = None
    z = None

    return z, W_z
    

In [ ]:
x2_w = np.vstack([6., 4., 2.1])
Qx2_w = np.diag([0.20, 0.09, 0.03])

z2_p_r = np.vstack([4., .3])
Wz2_p_r = np.diag([0.25, 0.04])

z1_w, Qz1 = to_world_frame(x1_w, Qx1_w, z1_p_r, W1)
z2_w, Qz2 = to_world_frame(x2_w, Qx2_w, z2_p_r, Wz2_p_r)

# Show results
fig, ax = plt.subplots()
plt.xlim([-.5, 10])
plt.ylim([-.5, 10])
plt.grid()
plt.tight_layout()

fig.canvas.draw()

DrawRobot(fig, ax, None, label='R1', color='blue')
PlotEllipse(fig, ax, x1_w, Qx1_w, color='blue')

DrawRobot(fig, ax, None, label='R2', color='green')
PlotEllipse(fig, ax, x2_w, Qx2_w, color='green')
   
ax.plot(None, None, 'o', label='Z1', color='blue')
PlotEllipse(fig, ax, z1_w, Qz1, color='blue')
          
ax.plot(None, None, 'o', label='Z2', color='green')
PlotEllipse(fig, ax, z2_w, Qz2, color='green')

z_w, Wz_w = combine_pdfs(z1_w, Qz1, z2_w, Qz2)
ax.plot(z_w[0, 0], z_w[1, 0], 'o', label='Z3', color='red')
PlotEllipse(fig, ax, z_w, Wz_w, color='red')
          
plt.legend()

# Print results
print( '----\tExercise 4.1.3\t----\n'+
    'z2_w = {}\'\n'.format(z2_w.flatten())+
    'Qz2 = \n{}\n'.format(Qz2)
    )

# Print results
print( '----\tExercise 4.1.3 part 2\t----\n'+
    'z_w = {}\'\n'.format(z_w.flatten())+
    'Wz_w = \n{}\n'.format(Wz_w)
    )

<span style="color:blue">Expected outputs:</span>

### Sensor measurement from x2

```
---- Exercise 4.1.3 ----
z2_w = [3.05042514 6.70185272]'
Qz2 = 
[[0.84693794 0.4333316 ]
 [0.4333316  0.81306206]]
```

### Combined information
```
---- Exercise 4.1.3 part 2 ----
z_w = [2.58757252 6.15534036]'
Wz_w = 
[[0.37966125 0.07773125]
 [0.07773125 0.36999739]]
```

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Having completed the code above, you will be able to **answer the following questions**:

- When working with landmarks, why do we ignore the information regarding orientation?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- In the two first assignments we computed the covariance matrix $\Sigma_{\mathbf{m}^1}$ of the landmark observed by robot $R1$ in two different cases: when the $R1$ pose was perfectly known, and having some uncertainty about it. Which covariance matrix was bigger? Is it bigger than that of the robot? Why? 

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- In the last assignment we got two different pdfs associated to the same landmark. Is that a contradiction? How did you manage to combine these two *pieces of information*?      

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## <font color="orange"><b><i>AI Appendix</i></b></font>

### A.1 How I used AI
- **Model/tool:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I asked AI to help with (bullets):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **Best prompt I used (paste):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I kept vs. changed from the AI output:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### A.2 AI review 
Ask one generative AI to review your memo using this prompt:

```
Act as a technical reviewer of this robotics notebook. Read my memo and return exactly 5 bullet points:

- two on theory (key ideas),
- two on practice (relevant programming concepts of python used, not particular functions or variables),
- one on how to improve the code (most critical aspect: limitations, parameter choices, runtime/robustness, possible bugs).

Keep each bullet ≤15 words and make them specific to my work.
```

Paste its **5** items here:

- **(Theory 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Theory 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Improvement)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

Finally, **do you agree with them?**, **would you have selected others?** 
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## <span style="color:green">OPTIONAL PARTS</span>

### <span style="color:green">OPTIONAL: Getting the relative pose between two robots</span>

Another robot `R2` is at pose $\mathbf{x}_2 \sim \mathcal{N}(\bar{\mathbf{x}}_2, \Sigma_{\mathbf{x}_2})$, with $\bar{\mathbf{x}}_2 = [6\text{ m}, 4\text{ m}, 2.1\text{ rad}]^T$ and $\Sigma_{\mathbf{x}_2} = \text{diag}([0.20, 0.09, 0.03])$. Plot $\mathbf{x}_2$ and its ellipse (covariance) in green ($\sigma = 1$). **Compute the relative pose $\mathbf{x}_{12}$ between `R1` and `R2`**, including its associated uncertainty $\Sigma_{\mathbf{x}_{12}}$. This scenario is shown in Fig. 8.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/assignment_3_scenario.png"/>
    <figcaption>Fig. 8. Illustration of the scenario in this assignment.</figcaption>
</center>

This relative pose can be obtained in two different ways:

- **Through the composition of poses**, but using $\ominus \mathbf{x}_1$ instead of $\mathbf{x}_1$. Implement it in ``inverse_composition1()``.

Mean, being $\ominus \mathbf{x}_1 = [x_{\ominus 1}, y_{\ominus 1}, \theta_{\ominus 1}]^T$ and $\mathbf{x}_2 = [x_2, y_2, \theta_2]^T$:

$$
\mathbf{x}_{12} = \ominus \mathbf{x}_1 \oplus \mathbf{x}_2 = \mathbf{f}(\ominus \mathbf{x}_1, \mathbf{x}_2) =
\begin{bmatrix}
 x_{\ominus 1} + x_2 \cos \theta_{\ominus 1} - y_2 \sin \theta_{\ominus 1} \\
 y_{\ominus 1} + x_2 \sin \theta_{\ominus 1} + y_2 \cos \theta_{\ominus 1} \\
 \theta_{\ominus 1} + \theta_2
\end{bmatrix}
$$

Covariance:

$$
\Sigma_{\mathbf{x}_{12}} = \frac{\partial \mathbf{x}_{12}}{\partial \ominus \mathbf{x}_1} \frac{\partial \ominus \mathbf{x}_1}{\partial \mathbf{x}_1} \Sigma_{\mathbf{x}_1} \left( \frac{\partial \ominus \mathbf{x}_1}{\partial \mathbf{x}_1} \right)^T \left( \frac{\partial \mathbf{x}_{12}}{\partial \ominus \mathbf{x}_1} \right)^T
+
\frac{\partial \mathbf{x}_{12}}{\partial \mathbf{x}_2} \Sigma_{\mathbf{x}_2} \left( \frac{\partial \mathbf{x}_{12}}{\partial \mathbf{x}_2} \right)^T
$$

Applying the chain rule, it can be rewritten as:

$$
\Sigma_{\mathbf{x}_{12}} = \frac{\partial \mathbf{x}_{12}}{\partial \ominus \mathbf{x}_1} \Sigma_{\ominus \mathbf{x}_1} \left( \frac{\partial \mathbf{x}_{12}}{\partial \ominus \mathbf{x}_1} \right)^T
+
\frac{\partial \mathbf{x}_{12}}{\partial \mathbf{x}_2} \Sigma_{\mathbf{x}_2} \left( \frac{\partial \mathbf{x}_{12}}{\partial \mathbf{x}_2} \right)^T
$$

Being:

$$
\frac{\partial \mathbf{x}_{12}}{\partial \ominus \mathbf{x}_1} =
\begin{bmatrix}
1 & 0 & -x_2 \sin \theta_{\ominus 1} - y_2 \cos \theta_{\ominus 1} \\
0 & 1 & x_2 \cos \theta_{\ominus 1} - y_2 \sin \theta_{\ominus 1} \\
0 & 0 & 1
\end{bmatrix}
\; \; \; \; \; \frac{\partial \mathbf{x}_{12}}{\partial \mathbf{x}_2} =
\begin{bmatrix}
\cos \theta_{\ominus 1} & -\sin \theta_{\ominus 1} & 0\\
\sin \theta_{\ominus 1} & \cos \theta_{\ominus 1} & 0\\
0 & 0 & 1
\end{bmatrix}
$$

$$
\frac{\partial \ominus \mathbf{x}_1}{\partial \mathbf{x}_1} =
\begin{bmatrix}
-\cos \theta_1 & -\sin \theta_1 & x_1 \sin \theta_1 - y_1 \cos \theta_1 \\
\sin \theta_1 & -\cos \theta_1 & x_1 \cos \theta_1 + y_1 \sin \theta_1 \\
0 & 0 & -1
\end{bmatrix}
\; \; \; \; \; \Sigma_{\ominus \mathbf{x}_1} = \frac{\partial \ominus \mathbf{x}_1}{\partial \mathbf{x}_1} \Sigma_{\mathbf{x}_1} \left( \frac{\partial \ominus \mathbf{x}_1}{\partial \mathbf{x}_1} \right)^T
$$

where $\mathbf{x}_1 = [x_1, y_1, \theta_1]^T$. $\\[5pt]$

- **Using the inverse composition of poses**, that is $\mathbf{x}_{12} = \ominus \mathbf{x}_1 \oplus \mathbf{x}_2 = \mathbf{x}_2 \ominus \mathbf{x}_1$. This one is given for you in ``inverse_composition2()``.

In [ ]:
def inverse_composition1(x1_w, Qx1_w, x2_w, Qx2_w):
    jac_inv_x = jac_tinv(x1_w)

    inv_r1 = (
        tinv(x1_w),
        jac_inv_x @ Qx1_w @ jac_inv_x.T
    )

    jac_x12_inv = J1(inv_r1[0], x2_w)
    jac_x12_x2 = J2(inv_r1[0], x2_w)

    x12_w = tcomp(None, None)
        
    Qx12_w = (
            None@inv_r1[1]@None 
            + None@Qx2_w@None
        )
    
    return x12_w, Qx12_w

In [ ]:
def inverse_composition2(x1_w, Qx1_w, x2_w, Qx2_w):
    dx, dy = x2_w[0, 0]-x1_w[0, 0], x2_w[1, 0]-x1_w[1, 0]
    a = x2_w[2, 0] - x1_w[2, 0]
    c, s = np.cos(x1_w[2, 0]), np.sin(x1_w[2, 0])

    x12_w = np.array([
        [dx*c + dy*s],
        [-dx*s + dy*c],
        [a]])
    
    jac_x12_r1 = np.array([
        [-c, -s, -dx*s + dy*c],
        [s, -c, -dx*c - dy*s],
        [0, 0, -1]
    ])

    jac_x12_r2 = np.array([
        [c, s, 0],
        [-s, c, 0],
        [0, 0, 1]
    ])

    Qx12_w = jac_x12_r1@Qx1_w@jac_x12_r1.T + jac_x12_r2@Qx2_w@jac_x12_r2.T

    return x12_w, Qx12_w

In [ ]:
# Robot R1
x1_w = np.vstack([1., 2., 0.5])
Qx1_w = np.diag([0.08, 0.6, 0.02])

# Robot R2
x2_w = np.vstack([6., 4., 2.1])
Qx2_w = np.diag([0.20, 0.09, 0.03])

# Obtain the relative pose x12 between both robots through the composition of poses
x12_w, Qx12_w = inverse_composition1(x1_w, Qx1_w, x2_w, Qx2_w)
print( '----\tWith method 1\t----\n'+
        'x12_w = {}\'\n'.format(x12_w.flatten())+
        'Qx12_w = \n{}\n'.format(Qx12_w))

# Obtain the relative pose x12 between both robots through the inverse composition of poses
x12_w, Qx12_w = inverse_composition2(x1_w, Qx1_w, x2_w, Qx2_w)
print( '----\tWith method 2\t----\n'+
        'x12_w = {}\'\n'.format(x12_w.flatten())+
        'Qx12_w = \n{}\n'.format(Qx12_w))

<span style="color:blue">Expected results:</span>
 ```
 x12_w = [ 5.34676389 -0.64196257  1.6       ]'
 
 Qx12_w = 
 [[0.38248035 0.24115    0.01283925]
 [0.24115    1.16751965 0.10693528]
 [0.01283925 0.10693528 0.05      ]]
 ```

<span style="color:green">***END OF OPTIONAL PART***</span>

### <span style="color:green">OPTIONAL: Predicting an observation from the second robot</span>

According to the information (provided by R1) that we have about the position of the landmark $\mathbf{m}$ in the world coordinates (its location $\bar{\mathbf{m}}^1 = [m_x, m_y]^T$ and its associated uncertainty $\Sigma_{\mathbf{m}^1}$), compute the *predicted observation* $\hat{\mathbf{z}}^2_p = [\hat{r}, \hat{\alpha}]^T$ that a range-bearing sensor mounted on `R2` would provide, along with its covariance $\Sigma_{\hat{\mathbf{z}}^2_p}$. The image below shows this scenario.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_sensing/images/assignment_4_scenario.png"/>
    <figcaption>Fig. 9. Illustration of the scenario in this practical case.</figcaption>
</center>


To do so, recall that the range-bearing model for taking measurements from the pose $\mathbf{x}_2 = [x_2, y_2, \theta_2]^T$ of `R2` is $\mathbf{z}^2_p = \mathbf{h}(\mathbf{x}_2, \mathbf{m}) + \mathbf{w}$. At this point, ignore the noise $\mathbf{w}$, so the predicted observation is *(Note: use [`np.arctan2()`](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html) for computing the angle)*:

$$
\hat{\mathbf{z}}^2_p = \begin{bmatrix} \hat{r} \\ \hat{\alpha} \end{bmatrix} =
\mathbf{h}(\mathbf{x}_2, \bar{\mathbf{m}}^1) =
\begin{bmatrix} \sqrt{(m_x-x_2)^2+(m_y-y_2)^2} \\ \text{atan}\left(\frac{m_y-y_2}{m_x-x_2}\right) - \theta_2 \end{bmatrix}
$$

Then, you need to compute the covariance of the predicted observation in polar coordinates, $\Sigma_{\hat{\mathbf{z}}^2_p}$. For that, first compute its covariance in cartesian coordinates in the `R2` frame, where the predicted observation is $\hat{\mathbf{z}}^2_c = \ominus \mathbf{x}_2 \oplus \mathbf{m}^1$:

$$
\Sigma_{\hat{\mathbf{z}}^2_c} = \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \ominus \mathbf{x}_2} \frac{\partial \ominus \mathbf{x}_2}{\partial \mathbf{x}_2} \ \Sigma_{\mathbf{x}_2} \ \left( \frac{\partial \ominus \mathbf{x}_2}{\partial \mathbf{x}_2} \right)^T \left( \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \ominus \mathbf{x}_2} \right)^T +
\frac{\partial \hat{\mathbf{z}}^2_c}{\partial \mathbf{m}^1} \ \Sigma_{\mathbf{m}^1} \ \left( \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \mathbf{m}^1} \right)^T
$$

Applying the chain rule, it can be rewritten as:

$$
\Sigma_{\hat{\mathbf{z}}^2_c} = \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \ominus \mathbf{x}_2} \Sigma_{\ominus \mathbf{x}_2} \left( \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \ominus \mathbf{x}_2} \right)^T + \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \mathbf{m}^1} \ \Sigma_{\mathbf{m}^1} \ \left( \frac{\partial \hat{\mathbf{z}}^2_c}{\partial \mathbf{m}^1} \right)^T
$$

Once you have the covariance expressed in cartesian coordinates, you can express it in polar coordinates by means of the following Jacobian:

$$
\frac{\partial \hat{\mathbf{z}}^2_p}{\partial \hat{\mathbf{z}}^2_c} =
\begin{bmatrix}
    \cos{\hat{\alpha}}  & \sin{\hat{\alpha}} \\
    -\sin{\hat{\alpha}} / \hat{r}  & \cos{\hat{\alpha}} / \hat{r}
\end{bmatrix}
$$

In [ ]:
def predicted_obs_from_pov(x1_w, Qx1_w, z1_w, Wz1_w):
    """ Method to translate a pose+covariance in the world frame to an observation.
    
        This method only translates the landmark to the pov of the robot.
        It does not simulate a new observation.
        
        Args:
            x1_w: Pose of the robot which acts as pov (x_2 in the text)
            Qx1_w: Covariance of the robot (Sigma_x_2)
            z1_w: Landmark observed in cartesian coordinates (world frame) (m^1)
            Wz1_w: Covariance associated to the landmark (Sigma_m^1).
        Returns:
            z2_pr: Predicted observation of z1_w from pov of x1_w (polar coordinates)
            W2_p: Covariance associated to z2_pr
    """

    # Take a measurement using the range-bearing model
    z2_pr = np.vstack([
            None, # distance
            None # angle
        ])
    
    # Obtain the uncertainty in the R2 reference frame using the composition of a pose and a landmark: 
    z1_ext = np.vstack([z1_w, 0]) # Prepare position and uncertainty shapes to the ones expected by inverse_composition
    Wz1_w_ext = np.pad(Wz1_w, [(0, 1), (0, 1)], mode='constant')
    _, Wz1_r = inverse_composition1(None, None, None, None)
    W2_c = Wz1_r[0:2,0:2]
      
    # Jacobian from cartesian to polar at z2p_r
    theta = z2_pr[1, 0]
    s, c = np.sin(theta), np.cos(theta)
    r = z2_pr[0, 0]

    Jac_car_pol = np.array([
        [None, None],
        [None, None]
    ])

    # Finally, propagate the uncertainty to polar coordinates in the
    # robot frame
    W2_p = None@None@None
    
    return z2_pr, W2_p

In [ ]:
x2_w = np.vstack([6., 4., 2.1])
Qx2_w = np.diag([0.20, 0.09, 0.03])

z2_pr, W2_p = predicted_obs_from_pov(x2_w, Qx2_w, z1_w, Wz1)
print( '---- Results ----\n'+
    'z2p_r = {}\'\n'.format(z2_pr.flatten())+
    'W2_p = \n{}\n'.format(W2_p)    
)

<span style="color:blue">Expected output:</span>
```
---- Results ----
z2p_r = [3.94880545 0.58862004]'
W2_p = 
[[ 1.31374309 -0.02614814]
 [-0.02614814  0.08555399]]
```

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

 - When predicting an observation of $\mathbf{m}$ from the second robot $R2$, why did we need to use the Jacobian $\partial \hat{\mathbf{z}}^2_p / \partial \hat{\mathbf{z}}^2_c$?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

<span style="color:green">***END OF OPTIONAL PART***</span>

## <span style="color:green">OPTIONAL</span>

<span style="color:green">As commented, a number of sensors can be mounted on a mobile robot. In the robotic sensing lecture we discussed some of the most popular ones. As an optional exercise, you can look for interesting information about any of them (or any one not listed below) and further describe it here to complete your knowledge.</span>

- Beacons
  - GPS
- Range sensors
  - Sonar
  - Infrared
  - Laser scanner
- Cameras
- RGB-D cameras

<span style="color:green">***END OF OPTIONAL PART***</span>

## <span style="color:green">OPTIONAL</span>

<span style="color:green">An alternative to *landmark observation models* are *scan observation* ones, which work with scan-based sensors. Below, the three most popular ones are listed. Surf the internet for some code illustrating any of them, and include it in the notebook with a brief description of how it works and its purpose. You could also implement an example using these models.</span>

## Scan observation models

Scan observation models $p(\mathbf{z}|\mathbf{x},\mathbf{m})$ are used when the sensor mounted on the robot provides a scan $\mathbf{z} = [z_1, \dots, z_K]^T$ measuring distance and angle to obstacles in the workspace, *e.g.* a laser range finder. In this case, the map is given as a discrete representation (typically, a grid map, where each cell is described by its position and whether it is free of obstacles or occupied, or a segment-based map), and data association is not explicitly addressed. 

#### Beam model
#### Likelihood field
#### Scan matching

<span style="color:green">***END OF OPTIONAL PART***</span>